# Sub-Task 2 — EDA & Algorithm Validation
**Telecom AI Platform | Gold Feature Store**

This notebook implements all steps required by Sub-Task 2:
1. Synthetic Gold data generation (proxy for real feature store)
2. Pearson + Spearman correlation matrix
3. Variance Inflation Factor (VIF) check
4. UMAP vs PCA dimensionality reduction benchmark
5. Axis 1 baseline — IsolationForest + Cloudflare agreement check
6. Axis 2 baseline — HDBSCAN + Silhouette Score
7. Axis 3 baseline — K-Means++ + Silhouette Score (sweep if needed)
8. Decision table → `eda-findings.md`

> **LGPD note:** All data is spatially aggregated at H3 hexagon level; no individual records exist.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make sure the project root is on sys.path when running from notebooks/
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ml.eda_utils import (
    compute_correlation_matrix,
    compute_vif,
    benchmark_dim_reduction,
    X_HI_COLUMNS,
)
from ml.baseline_runners import run_all_baselines

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
logger = logging.getLogger('eda_notebook')

print('Environment ready. Project root:', project_root)

---
## 1. Load / Generate Gold Feature Data

When the real Gold Parquet is available under `data/gold/`, point `GOLD_PATH` to it.
Until then, synthetic data is generated with realistic Brazilian telecom distributions
so the full EDA + validation pipeline can be exercised end-to-end.

In [ ]:
GOLD_PATH = project_root / 'data' / 'gold'
USE_SYNTHETIC = True  # Set False once real data is available

if not USE_SYNTHETIC and GOLD_PATH.exists():
    parquet_files = list(GOLD_PATH.rglob('*.parquet'))
    gold_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
    logger.info('Loaded %d rows from Gold Parquet (%d files).', len(gold_df), len(parquet_files))
else:
    logger.info('Generating synthetic Gold feature data …')
    np.random.seed(42)
    N = 2_000  # Number of synthetic hexagons

    # --- Synthetic distributions tuned to Brazilian telecom reality ---
    # Three latent market segments to make clustering meaningful
    segment = np.random.choice(['premium', 'mid_risk', 'essential'], size=N, p=[0.25, 0.35, 0.40])

    def seg(prem, mid, ess):
        return np.where(segment == 'premium', prem, np.where(segment == 'mid_risk', mid, ess))

    d_down    = np.clip(np.random.normal(seg(120, 45, 18), seg(30, 15, 8)), 1, 500)
    d_up      = np.clip(d_down * np.random.uniform(0.3, 0.6, N) + np.random.normal(0, 2, N), 0.5, 200)
    lat       = np.clip(np.random.normal(seg(15, 35, 60), seg(5, 10, 20)), 5, 200)
    vol_test  = np.clip(np.random.normal(seg(8, 5, 3), 1.5), 1, 30)
    n_dom     = np.clip(np.random.negative_binomial(seg(80, 50, 30), 0.4, N).astype(float), 1, 2000)
    n_com     = np.clip((n_dom * np.random.uniform(0.1, 0.4, N)).astype(float), 0, 500)
    ren_med   = np.clip(np.random.normal(seg(4200, 1800, 950), seg(600, 400, 200)), 300, 10000)
    n_erb     = np.clip(np.random.poisson(seg(12, 6, 3), N).astype(float), 0, 80)
    HHI       = np.clip(np.random.normal(seg(2000, 4200, 5500), seg(400, 500, 600)), 0, 10000)
    vol_reclam = np.clip(np.random.exponential(seg(3, 8, 15), N), 0, 80)

    # Introduce mild correlation: high HHI → more complaints
    vol_reclam += 0.002 * (HHI - 3000).clip(min=0)
    # High download speed → lower latency (moderate negative correlation)
    lat -= 0.05 * d_down.clip(max=100)
    lat = np.clip(lat, 5, 200)

    # Inject ~3% anomalies for Axis 1 testing
    n_anomalies = int(N * 0.03)
    anomaly_idx = np.random.choice(N, n_anomalies, replace=False)
    lat[anomaly_idx]        += np.random.uniform(80, 150, n_anomalies)
    vol_reclam[anomaly_idx] += np.random.uniform(30, 60, n_anomalies)
    d_down[anomaly_idx]     *= 0.1

    h3_ids = [f'8a2a{i:07x}fffff' for i in range(N)]
    anomalous_h3_ids = set(h3_ids[i] for i in anomaly_idx)

    gold_df = pd.DataFrame({
        'h3_id':     h3_ids,
        'd_down':    d_down,
        'd_up':      d_up,
        'lat':       lat,
        'vol_test':  vol_test,
        'n_dom':     n_dom,
        'n_com':     n_com,
        'ren_med':   ren_med,
        'n_erb':     n_erb,
        'HHI':       HHI,
        'vol_reclam': vol_reclam,
        '_segment':  segment,  # ground-truth label for validation only
    })

    logger.info('Synthetic Gold: %d hexagons | %d injected anomalies', N, n_anomalies)

gold_df.describe().round(2)

---
## 2. Pearson & Spearman Correlation Matrix

In [ ]:
pearson_corr  = compute_correlation_matrix(gold_df, method='pearson')
spearman_corr = compute_correlation_matrix(gold_df, method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, corr, title in zip(axes, [pearson_corr, spearman_corr], ['Pearson', 'Spearman']):
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(
        corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
        vmin=-1, vmax=1, center=0, ax=ax, square=True,
        linewidths=0.5, cbar_kws={'shrink': 0.8}
    )
    ax.set_title(f'{title} Correlation — x_hi Features', fontsize=13)
plt.tight_layout()
plt.savefig(project_root / 'notebooks' / 'fig_correlation_matrix.png', dpi=120)
plt.show()

# Flag strong correlations (|r| > 0.7)
strong_pearson = [
    (i, j, round(pearson_corr.loc[i, j], 3))
    for i in pearson_corr.index for j in pearson_corr.columns
    if i < j and abs(pearson_corr.loc[i, j]) > 0.70
]
print('\nStrong Pearson correlations (|r| > 0.7):')
for a, b, r in strong_pearson:
    print(f'  {a} ↔ {b}: r={r}')
if not strong_pearson:
    print('  None detected.')

---
## 3. Variance Inflation Factor (VIF)

In [ ]:
vif_df = compute_vif(gold_df)
print(vif_df.to_string(index=False))

flagged = vif_df[vif_df['flag_multicollinearity']]
if len(flagged):
    print(f'\n⚠️  {len(flagged)} feature(s) with VIF > 10 (multicollinearity concern):')
    print(flagged[['feature', 'VIF']].to_string(index=False))
else:
    print('\n✅ All features: VIF ≤ 10 — no multicollinearity detected.')

---
## 4. Dimensionality Reduction Benchmark: UMAP vs PCA

In [ ]:
dim_bench = benchmark_dim_reduction(gold_df)

pca_res  = dim_bench['pca']
umap_res = dim_bench['umap']

print(f"PCA  — Total explained variance (2 components): {pca_res['total_variance']:.3f}")
print(f"       PC1={pca_res['explained_variance'][0]:.3f} | PC2={pca_res['explained_variance'][1]:.3f}")
print(f"UMAP — Trustworthiness: {umap_res['trustworthiness']:.4f}")
print(f"\n→ Recommended method: {dim_bench['recommendation']}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
segment_labels = gold_df['_segment'].values if '_segment' in gold_df.columns else None
palette = {'premium': '#1f77b4', 'mid_risk': '#ff7f0e', 'essential': '#2ca02c'}

for ax, key, title in zip(axes,
                           ['pca',  'umap'],
                           [f'PCA (var={pca_res["total_variance"]:.2%})',
                            f'UMAP (trust={umap_res["trustworthiness"]:.4f})']):
    emb = dim_bench[key]['embedding']
    if segment_labels is not None:
        for seg, color in palette.items():
            mask = segment_labels == seg
            ax.scatter(emb[mask, 0], emb[mask, 1], c=color, label=seg,
                       alpha=0.4, s=8, rasterized=True)
        ax.legend(markerscale=3, title='Segment')
    else:
        ax.scatter(emb[:, 0], emb[:, 1], alpha=0.3, s=8, rasterized=True)
    ax.set_title(title)
    ax.set_xlabel('Dim 1')
    ax.set_ylabel('Dim 2')

plt.suptitle('Dimensionality Reduction Benchmark — Gold Feature Store', fontsize=13)
plt.tight_layout()
plt.savefig(project_root / 'notebooks' / 'fig_dim_reduction_benchmark.png', dpi=120)
plt.show()

---
## 5–7. Baseline Model Validation (All Three Axes)

In [ ]:
# Provide Cloudflare ground-truth if generated synthetically above
cf_outages = anomalous_h3_ids if USE_SYNTHETIC else None  # Replace with real set when available

baseline_results = run_all_baselines(gold_df, cloudflare_outage_h3_ids=cf_outages)

print('\n' + '='*70)
print('BASELINE VALIDATION SUMMARY')
print('='*70)
for res in baseline_results:
    print(f"\nAxis {res['axis']}")
    print(f"  Algorithm : {res['algorithm']}")
    print(f"  Metric    : {res['metric_name']} = {res['metric_value']:.4f}")
    print(f"  Decision  : {res['decision']}")
    print(f"  Rationale : {res['rationale']}")

---
## 8. Persist `eda-findings.md` Decision Table

In [ ]:
from datetime import date

axis_labels = {
    '1_manutencao_e_criticidade':    'Axis 1 — Maintenance & Criticality',
    '2_novos_negocios_e_expansao':   'Axis 2 — Expansion Opportunities',
    '3_regioes_propicias_para_promocoes': 'Axis 3 — Promotional Targeting',
}

vif_flagged = vif_df[vif_df['flag_multicollinearity']]['feature'].tolist()
dim_rec     = dim_bench['recommendation']
pca_var     = pca_res['total_variance']
umap_trust  = umap_res['trustworthiness']

lines = [
    '# EDA Findings — Sub-Task 2',
    f'',
    f'**Generated:** {date.today().isoformat()}  ',
    f'**Source:** `notebooks/01_eda_feature_store.ipynb`  ',
    f'**Feature vector:** `x_hi = [d_down, d_up, lat, vol_test, n_dom, n_com, ren_med, n_erb, HHI, vol_reclam]`',
    '',
    '---',
    '',
    '## 1. Multicollinearity Analysis',
    '',
    '### 1.1 Pearson & Spearman Correlation',
    '',
]

if strong_pearson:
    lines += [
        f'Strong Pearson correlations (|r| > 0.70) detected:',
        '',
        '| Feature A | Feature B | Pearson r |',
        '|---|---|---|',
    ] + [f'| {a} | {b} | {r} |' for a, b, r in strong_pearson]
else:
    lines += ['No strong pairwise correlations (|r| > 0.70) detected.']

lines += ['', '### 1.2 Variance Inflation Factor (VIF)', '', '| Feature | VIF | Multicollinearity Flag |', '|---|---|---|']
for _, row in vif_df.iterrows():
    flag = '⚠️ YES' if row['flag_multicollinearity'] else 'NO'
    lines.append(f"| {row['feature']} | {row['VIF']:.2f} | {flag} |")

if vif_flagged:
    lines += ['', f'**VIF > 10 flags:** {vif_flagged}. Consider removing or combining these features before production pipelines.']
else:
    lines += ['', '**No VIF > 10 flags.** All features can be retained in the production feature vector.']

lines += [
    '',
    '---',
    '',
    '## 2. Dimensionality Reduction Benchmark: UMAP vs PCA',
    '',
    '| Method | Metric | Value |',
    '|---|---|---|',
    f'| PCA | Cumulative explained variance (2 components) | {pca_var:.4f} |',
    f'| UMAP | Trustworthiness (local neighbourhood preservation) | {umap_trust:.4f} |',
    '',
    f'**Recommendation: {dim_rec}**  ',
    f'Rationale: ',
]
if dim_rec == 'UMAP':
    lines.append(f'UMAP trustworthiness ({umap_trust:.4f}) > 0.90 AND PCA explains only {pca_var:.1%} of variance — UMAP better preserves the nonlinear spatial structure of the telecom data. Apply UMAP as a pre-processing step for HDBSCAN (Axis 2) and optionally for IsolationForest (Axis 1).')
else:
    lines.append(f'PCA explains {pca_var:.1%} of variance in 2 components. UMAP trustworthiness ({umap_trust:.4f}) does not clearly exceed PCA — PCA is sufficient for Axis 1/3 and optional for Axis 2.')

lines += [
    '',
    '---',
    '',
    '## 3. Baseline Model Decision Table',
    '',
    '| Axis | Algorithm | Metric | Score | Decision | Rationale |',
    '|---|---|---|---|---|---|',
]

for res in baseline_results:
    label = axis_labels.get(res['axis'], res['axis'])
    lines.append(
        f"| {label} | {res['algorithm']} | {res['metric_name']} "
        f"| {res['metric_value']:.4f} | {res['decision']} | {res['rationale']} |"
    )

# Extra detail per axis
lines += ['', '---', '', '## 4. Per-Axis Detail', '']
for res in baseline_results:
    label = axis_labels.get(res['axis'], res['axis'])
    lines += [f'### {label}', '']
    extra = res['extra']
    lines += [f'- **Features used:** `{extra["features_used"]}`']
    if res['axis'] == '1_manutencao_e_criticidade':
        lines += [
            f"- **Anomaly rate:** {extra['anomaly_rate']:.2%} ({extra['n_anomalies']} / {extra['n_total']} hexagons)",
            f"- **Anomaly score threshold:** 0.65 (per `knowledge-api.json` spec)",
        ]
    elif res['axis'] == '2_novos_negocios_e_expansao':
        lines += [
            f"- **n_clusters:** {extra['n_clusters']}",
            f"- **Noise rate:** {extra['noise_rate']:.2%}",
            f"- **Criteria concentration:** {f\"{extra['criteria_concentration']:.2%}\" if extra['criteria_concentration'] is not None else 'N/A'}",
        ]
    elif res['axis'] == '3_regioes_propicias_para_promocoes':
        lines += [
            f"- **Initial k:** {extra['initial_k']} | **Best k:** {extra['best_k']}",
            f"- **k sweep performed:** {extra['swept']}",
        ]
        if extra['cluster_stats']:
            lines += ['', '**Cluster profiles:**', '', '| Cluster | Size | mean_ren_med | mean_HHI | mean_d_up | mean_vol_reclam |', '|---|---|---|---|---|---|']
            for cs in extra['cluster_stats']:
                lines.append(
                    f"| {cs['cluster_id']} | {cs['size']} "
                    f"| {cs.get('mean_ren_med', 'N/A')} "
                    f"| {cs.get('mean_HHI', 'N/A')} "
                    f"| {cs.get('mean_d_up', 'N/A')} "
                    f"| {cs.get('mean_vol_reclam', 'N/A')} |"
                )
    lines.append('')

lines += [
    '---',
    '',
    '## 5. Actions and knowledge-api.json Updates',
    '',
]

adjustments_needed = [r for r in baseline_results if r['decision'] == 'ADJUST']
if adjustments_needed:
    lines += ['The following axes require adjustments before production implementation:', '']
    for res in adjustments_needed:
        label = axis_labels.get(res['axis'], res['axis'])
        lines += [f'- **{label}:** {res["rationale"]}', '']
    lines += ['See `knowledge-api.json` for updated hyperparameters.']
else:
    lines += ['All three axes validated. No amendments required to `knowledge-api.json`.']

findings_path = project_root / 'eda-findings.md'
findings_path.write_text('\n'.join(lines), encoding='utf-8')
print(f'✅  Written → {findings_path}')

---
## Notebook Complete

All Sub-Task 2 artefacts have been produced:
- `eda-findings.md` — decision table and full EDA narrative
- `fig_correlation_matrix.png` — Pearson + Spearman heatmaps
- `fig_dim_reduction_benchmark.png` — UMAP vs PCA scatter plots

Next step: **Sub-Task 3 — ML Pipeline Implementation**